# Validate a Single-Step Command Job

Load the checked-in sample merge job, bind the configured cluster and its
managed identity, submit it, and verify completion and downloaded output.
Executing the submission cell always starts a billable job.

Python setup is defined in this notebook. Shared YAML, component scripts, and
sample data remain under `workshop`; this copy uses only its suite-local `.env`
and writes downloads under this suite.


In [ ]:
from pathlib import Path
from azure.ai.ml import MLClient, load_job
from azure.ai.ml.entities import ManagedIdentityConfiguration
from azure.identity import AzureCliCredential, ManagedIdentityCredential
from dotenv import dotenv_values

notebook_path = Path(globals().get("__vsc_ipynb_file__", Path.cwd() / "01_submit_single_step_merge.ipynb"))
SUITE_ROOT = notebook_path.resolve().parents[1]
env = dotenv_values(SUITE_ROOT / ".env", encoding="utf-8-sig", interpolate=False)

SUBSCRIPTION_ID = (env.get("AZURE_SUBSCRIPTION_ID") or "").strip()
RESOURCE_GROUP = (env.get("AZURE_RESOURCE_GROUP") or "").strip()
WORKSPACE_NAME = (env.get("AZUREML_WORKSPACE_NAME") or "").strip()
COMPUTE_NAME = (env.get("AZUREML_COMPUTE_NAME") or "").strip()
COMPUTE_IDENTITY_CLIENT_ID = (env.get("AZUREML_COMPUTE_IDENTITY_CLIENT_ID") or "").strip()
if not all((SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME, COMPUTE_NAME, COMPUTE_IDENTITY_CLIENT_ID)):
    raise ValueError(f"Fill workspace coordinates, compute name, and identity Client ID in {SUITE_ROOT / '.env'}.")

use_managed_identity = (env.get("USE_MANAGED_IDENTITY", "true") or "").strip().lower()
if use_managed_identity not in {"true", "false"}:
    raise ValueError("USE_MANAGED_IDENTITY must be true or false in .env.")
credential = ManagedIdentityCredential(process_timeout=60) if use_managed_identity == "true" else AzureCliCredential(process_timeout=60)
ml_client = MLClient(credential, SUBSCRIPTION_ID, RESOURCE_GROUP, WORKSPACE_NAME)
SAMPLE_ROOT = SUITE_ROOT.parents[1] / "workshop"
pipeline_path = SAMPLE_ROOT / "pipelines/single-step-merge-job.yaml"
print(f"Workspace: {WORKSPACE_NAME}; compute: {COMPUTE_NAME}")

In [ ]:
job = load_job(pipeline_path)
job.compute = COMPUTE_NAME
job.identity = ManagedIdentityConfiguration(client_id=COMPUTE_IDENTITY_CLIENT_ID)
job.display_name = "Readiness single-step taxi merge"
job.tags = {"validation": "workspace-readiness", "operation": "single-step-merge"}

print(f"Loaded: {type(job).__name__}")
print(f"Compute: {COMPUTE_NAME}")
print(f"Runtime identity: compute cluster UMI {COMPUTE_IDENTITY_CLIENT_ID}")
print("Prepared only; run the submission cell when ready.")


## Submit Once

This cell starts a billable job. Status checks and output downloads are separate and do not resubmit it.

In [ ]:
submitted_job = ml_client.jobs.create_or_update(job)
print(f"Submitted: {submitted_job.name}")

## Check Status

This is a status snapshot. To resume an existing run after a kernel restart, replace `submitted_job.name` with its quoted run name. The next cell waits for completion without SDK log streaming.

In [ ]:
JOB_NAME = submitted_job.name
final_job = ml_client.jobs.get(JOB_NAME)
print(f"Job: {final_job.name}; status: {final_job.status}")
print(f"Studio: {final_job.studio_url}")

## Download the Completed Output

Wait up to 20 minutes, checking every 10 seconds, then download only after `Completed`. A timeout does not cancel or resubmit the job; rerun this cell to keep waiting. The SDK resolves the named output's artifact location.

In [ ]:
import time

deadline = time.monotonic() + 20 * 60
final_job = ml_client.jobs.get(JOB_NAME)
while final_job.status not in {"Completed", "Failed", "Canceled", "NotResponding"}:
    if time.monotonic() >= deadline:
        raise TimeoutError(f"Timed out after 20 minutes waiting for job {JOB_NAME}; last status: {final_job.status}.")
    time.sleep(10)
    final_job = ml_client.jobs.get(JOB_NAME)
if final_job.status in {"Failed", "Canceled", "NotResponding"}:
    raise RuntimeError(f"Job ended with status {final_job.status}; inspect its Studio page.")
if final_job.status == "Completed":
    download_dir = SUITE_ROOT / "outputs/single_step_merge" / final_job.name
    ml_client.jobs.download(
        final_job.name,
        output_name="merged_data",
        download_path=download_dir,
    )
    merged_path = download_dir / "named-outputs/merged_data/merged_taxi_data.csv"
    if not merged_path.is_file():
        raise FileNotFoundError(f"Expected downloaded output at {merged_path}")
    print(f"Downloaded merged data: {merged_path}")


## Expected Result

The command job completes on the configured cluster and publishes a merged taxi CSV as its `merged_data` output.

Next: `02_submit_integration_compare.ipynb`.